# 16 Core Implementation


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLP/lab09'
import os, json, random, numpy as np
os.makedirs(DRIVE_DIR, exist_ok=True)
SEED = 42
random.seed(SEED); np.random.seed(SEED)
!pip -q install datasets==3.6.0 transformers
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
ds=load_dataset('eriktks/conll2003', trust_remote_code=True)
label_names=ds['test'].features['ner_tags'].feature.names
model_dir=f'{DRIVE_DIR}/bert-base-cased-conll2003'
tokenizer=AutoTokenizer.from_pretrained(model_dir, use_fast=True)
model=AutoModelForTokenClassification.from_pretrained(model_dir)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); model.to(device).eval()
row=ds['test'][0]
enc=tokenizer(row['tokens'],is_split_into_words=True,truncation=True,max_length=256)
word_ids=enc.word_ids(); aligned=[]; previous=None
for wid in word_ids:
    aligned.append(-100 if wid is None or wid==previous else row['ner_tags'][wid])
    previous=wid
with torch.no_grad():
    batch={k:torch.tensor([v]).to(device) for k,v in enc.items()}
    logits=model(**batch).logits[0].cpu()
pred_ids=logits.argmax(-1).tolist(); word_predictions=[]; seen=set()
for wid,pred in zip(word_ids,pred_ids):
    if wid is not None and wid not in seen:
        word_predictions.append(label_names[pred]); seen.add(wid)
print('original_words:',row['tokens'])
print('tokens:',tokenizer.convert_ids_to_tokens(enc['input_ids']))
print('input_ids:',enc['input_ids'])
print('attention_mask:',enc['attention_mask'])
print('word_ids:',word_ids)
print('labels:',[('IGNORE' if i==-100 else label_names[i]) for i in aligned])
print('logits:',logits.tolist())
print('predictions_by_word:',word_predictions)
